### ЗАДАЧА: Пакетная обработка переводов между кошельками (exceptions + business rules)

Есть набор кошельков и список входящих переводов.
Нужно безопасно обработать пакет операций: валидные переводы применить к балансам,
ошибочные сохранить в отчёт, не останавливая всю обработку.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `TransferError`
   - `TransferFormatError`
   - `AccountNotFoundError`
   - `CurrencyMismatchError`
   - `InsufficientFundsError`
   - `TransferAmountError`.

2. Функцию `parse_transfer(raw)`:
   - формат строки: `transfer_id|from_user|to_user|amount`
   - `amount` должен быть числом и `> 0`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `apply_transfer(transfer, wallets)`:
   - проверить, что оба пользователя существуют
   - нельзя переводить самому себе
   - валюты кошельков отправителя и получателя должны совпадать
   - у отправителя должно хватать средств
   - при успехе обновить балансы в `wallets`
   - вернуть краткий словарь результата.

4. Функцию `process_batch(rows, wallets)`:
   - для каждой строки вызвать `parse_transfer`, потом `apply_transfer`
   - вернуть `(successes, errors)`
   - ошибки хранить как `(raw, error_type, message)`
   - не прерывать цикл на первой ошибке.

5. Вывести:
   - успешные переводы,
   - ошибки по типам,
   - итоговые балансы,
   - пользователя с максимальным балансом в валюте `USD`.


In [ ]:
wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]


class TransferError(Exception):
    pass


class TransferFormatError(TransferError):
    pass


class AccountNotFoundError(TransferError):
    pass


class CurrencyMismatchError(TransferError):
    pass


class InsufficientFundsError(TransferError):
    pass


class TransferAmountError(TransferError):
    pass


def parse_transfer(raw):
    # TODO: распарсить строку и вернуть dict перевода
    # TODO: при ошибке конвертации amount использовать raise ... from ...
    parts = raw.split('|')
    if len(parts) != 4:
        raise TransferFormatError("Некорректный формат строки")
    transaction_id, sender, receiver, amount_str = parts
    
    try:
        amount = float(amount_str)
    except ValueError as e:
        raise TransferAmountError(f"Некорректное значение суммы: {amount_str}") from e
    return {
        "transaction_id": transaction_id,
        "sender": sender,
        "receiver": receiver,
        "amount": amount
    }


def apply_transfer(transfer, wallets):
    # TODO: проверить существование аккаунтов
    # TODO: запретить перевод самому себе
    # TODO: проверить совпадение валют
    # TODO: проверить баланс отправителя
    # TODO: обновить балансы и вернуть dict результата
    sender = transfer["sender"]
    receiver = transfer["receiver"]
    amount = transfer["amount"]

    if sender not in wallets:
        raise AccountNotFoundError(f"Аккаунт {sender} не найден")
    if receiver not in wallets:
        raise AccountNotFoundError(f"Аккаунт {receiver} не найден")
    # 2. Запретить перевод самому себе
    if sender == receiver:
        raise TransferError("Перевод самому себе запрещен")
    sender_wallet = wallets[sender]
    receiver_wallet = wallets[receiver]

    if sender_wallet["currency"] != receiver_wallet["currency"]:
        raise CurrencyMismatchError("Валюта отправителя и получателя не совпадает")
    
    if sender_wallet["balance"] < amount:
        raise InsufficientFundsError("Недостаточно средств")
    
    sender_wallet["balance"] -= amount
    receiver_wallet["balance"] += amount

    return {
        "transaction_id": transfer["transaction_id"],
        "sender": sender,
        "receiver": receiver,
        "amount": amount,
        "sender_balance": sender_wallet["balance"],
        "receiver_balance": receiver_wallet["balance"]
    }

def process_batch(rows, wallets):
    # TODO: вернуть (successes, errors)
    successes = []
    errors = []
    for row in rows:
        try:
            transfer = parse_transfer(row)
            result = apply_transfer(transfer, wallets)
            successes.append(result)
        except TransferError as e:
            errors.append((row, str(e)))
    return successes, errors


# TODO: вызвать process_batch(rows, wallets)
successes, errors = process_batch(rows, wallets)

# TODO: вывести успешные переводы
print("Успешные переводы:")
for s in successes:
    print(s)

# TODO: вывести ошибки по типам
print("\nОшибки:")
for row, err_msg in errors:
    print(f"Строка: {row} — Ошибка: {err_msg}")

# TODO: вывести итоговые балансы
print("\nИтоговые балансы:")
for user, data in wallets.items():
    print(f"{user}: {data["balance"]} {data["currency"]}")

# TODO: найти richest_usd_user

richest_user = None
max_balance = -1
for user, data in wallets.items():
    if data['currency'] == 'USD' and data['balance'] > max_balance:
        max_balance = data['balance']
        richest_user = user

print(f"\nСамый богатый в USD: {richest_user} с балансом {max_balance}")